## Dictionary-Based Text Analysis in Python

This Jupyter Notebook demonstrates dictionary-based text analysis using a custom dataset.
It includes:
1. Loading a CSV dataset
2. Cleaning text
3. Applying a dictionary to count word occurrences
4. Using a preloaded dictionary
5. Summarizing dictionary matches

Make sure you have a CSV file with a text column before running this notebook.


In [ ]:
# Install required packages (uncomment if running for the first time)
# !pip install pandas nltk

In [1]:
# Import necessary libraries
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/deniseroth/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

### **1. Load Dataset**

Replace 'your_dataset.csv' with the actual file name.
Ensure the CSV has a column named 'text' containing textual data.


In [2]:
df = pd.read_csv("/Users/deniseroth/Documents/Downloads/mental_health_sentiment.csv")
print("Dataset loaded successfully!")
print(df.head())

Dataset loaded successfully!
  id                                               text sentiment  \
0  0                                         oh my gosh   Anxiety   
1  1  trouble sleeping, confused mind, restless hear...   Anxiety   
2  2  All wrong, back off dear, forward doubt. Stay ...   Anxiety   
3  3  I've shifted my focus to something else but I'...   Anxiety   
4  4  I'm restless and restless, it's been a month n...   Anxiety   

   condition_binary  
0                 1  
1                 1  
2                 1  
3                 1  
4                 1  


### **2. Clean Text**

Text cleaning involves:
- Converting to lowercase
- Removing special characters and punctuation
- Tokenizing text


In [3]:
def clean_text(text):
    text = str(text).lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove punctuation
    return word_tokenize(text)  # Tokenize

# Apply cleaning
df['cleaned_text'] = df['text'].apply(clean_text)
print("Text cleaned and tokenized successfully!")

Text cleaned and tokenized successfully!


### **3. Define a Dictionary for Analysis**

Create a dictionary of words and their categories.
Modify this dictionary based on your research needs.

In [4]:
dictionary = {
    "positive": ["good", "excellent", "happy", "great", "fantastic"],
    "negative": ["bad", "terrible", "horrible", "sad", "poor"],
}

### **4. Apply Dictionary to Text**

Count occurrences of dictionary words in each document.


In [5]:
def count_dictionary_words(tokens, category_words):
    return sum(1 for word in tokens if word in category_words)

In [6]:
# Apply dictionary categories
df['positive_count'] = df['cleaned_text'].apply(lambda x: count_dictionary_words(x, dictionary['positive']))
df['negative_count'] = df['cleaned_text'].apply(lambda x: count_dictionary_words(x, dictionary['negative']))

## **5. Inspect Dictionary Results**

After applying the dictionary, we can inspect the results:
- Which words from the dictionary were found in each text?
- View example texts with high positive or negative scores.

In [7]:
def extract_matching_words(tokens, category_words):
    """Return words from tokens that match the dictionary category."""
    return [word for word in tokens if word in category_words]

In [8]:
# Extract matching words for inspection
df['positive_words'] = df['cleaned_text'].apply(lambda x: extract_matching_words(x, dictionary['positive']))
df['negative_words'] = df['cleaned_text'].apply(lambda x: extract_matching_words(x, dictionary['negative']))

In [9]:
# Display sample results sorted by highest positive and negative scores
df_sorted = df.sort_values(by=['positive_count', 'negative_count'], ascending=[False, False])
df_inspection = df_sorted[['text', 'positive_words', 'negative_words', 'positive_count', 'negative_count']].head(10)
print("Sample dictionary matches with highest scores:")
print(df_inspection)

Sample dictionary matches with highest scores:
                                                    text  \
40745  Please help me understand what I went through ...   
9244   The title is not meant to discourage others, b...   
14876  I really just want to get this off my chest an...   
8240   Hello everyone,I rarely post on Reddit but.......   
27432  this is my first reddit post also my first tim...   
40454  good or bad person does anyone else struggle w...   
9381   Sorry this is long but I doubt anyone will eve...   
40146  What are you guys good at? Sometimes I forget ...   
41146  What are you guys good at? Sometimes I forget ...   
29101  it doesn t matter anymore i m going to copy an...   

                                          positive_words  \
40745  [good, happy, good, great, happy, great, great...   
9244   [happy, good, good, great, good, great, happy,...   
14876  [great, good, good, good, good, good, good, go...   
8240   [happy, happy, happy, happy, happy, happy, ha

### **6. Use a Preloaded Dictionary (LIWC Example)**

Instead of a manually created dictionary, you can load and use a predefined one.
NLTK provides built-in positive/negative word lists from the Opinion Lexicon.


In [10]:
from nltk.corpus import opinion_lexicon

In [11]:
nltk.download('opinion_lexicon')

[nltk_data] Downloading package opinion_lexicon to
[nltk_data]     /Users/deniseroth/nltk_data...
[nltk_data]   Package opinion_lexicon is already up-to-date!


True

In [12]:
# Load positive and negative words from the lexicon
positive_opinion = set(opinion_lexicon.positive())
negative_opinion = set(opinion_lexicon.negative())

In [13]:
# Print sample words
print("Positive words:", list(positive_opinion)[:10])
print("Negative words:", list(negative_opinion)[:10])

Positive words: ['heartfelt', 'convenience', 'refinement', 'rectify', 'tantalize', 'winners', 'yay', 'integral', 'affably', 'happiness']
Negative words: ['unjust', 'cataclysmal', 'hellion', 'assassinate', 'ulterior', 'bashed', 'liars', 'covetous', 'concerned', 'fragmented']


In [14]:
# Apply dictionary
def count_words_in_dict(tokens, word_set):
    return sum(1 for word in tokens if word in word_set)

df['positive_count_opinion'] = df['cleaned_text'].apply(lambda x: count_words_in_dict(x, positive_opinion))
df['negative_count_opinion'] = df['cleaned_text'].apply(lambda x: count_words_in_dict(x, negative_opinion))

### **7. Summarize Results**

View how many documents contain positive vs. negative words. 
Compare with our customized dictionary from earlier.
Also take a look at the documents that score high on positive words and negative words according to the preloaded dictionary.

In [15]:
print("Dictionary Analysis Results:")
print(df[['text', 'positive_count', 'negative_count', 'positive_count_opinion', 'negative_count_opinion']].head())

Dictionary Analysis Results:
                                                text  positive_count  \
0                                         oh my gosh               0   
1  trouble sleeping, confused mind, restless hear...               0   
2  All wrong, back off dear, forward doubt. Stay ...               0   
3  I've shifted my focus to something else but I'...               0   
4  I'm restless and restless, it's been a month n...               0   

   negative_count  positive_count_opinion  negative_count_opinion  
0               0                       0                       0  
1               0                       0                       3  
2               0                       0                       4  
3               0                       0                       1  
4               0                       0                       2  


In [16]:
# Extract matching words for inspection
df['positive_words_opinion'] = df['cleaned_text'].apply(lambda x: extract_matching_words(x, positive_opinion))
df['negative_words_opinion'] = df['cleaned_text'].apply(lambda x: extract_matching_words(x, negative_opinion))

In [17]:
# Display sample results sorted by highest positive and negative scores
df_sorted = df.sort_values(by=['positive_count_opinion', 'negative_count_opinion'], ascending=[False, False])
df_inspection = df_sorted[['text', 'positive_words_opinion', 'negative_words_opinion', 'positive_count_opinion', 'negative_count_opinion']].head(10)
print("Sample dictionary matches with highest scores:")
print(df_inspection)

Sample dictionary matches with highest scores:
                                                    text  \
40745  Please help me understand what I went through ...   
29101  it doesn t matter anymore i m going to copy an...   
36009  DEPRESSION HAS A PURPOSE: HOW TO USE IT RIGHT ...   
9265   I no longer know what else to do but write thi...   
9293   And has life gotten better?&amp;#x200B;No. Eve...   
14358  will i ever be noticed? is my life worth anyth...   
15596  I need support or encouragement. I (29M) reall...   
9381   Sorry this is long but I doubt anyone will eve...   
9244   The title is not meant to discourage others, b...   
28928  we ve been seeing a worrying increase in pro s...   

                                  positive_words_opinion  \
40745  [sensitive, right, work, worked, work, beautif...   
29101  [pride, proud, fine, right, unforgettable, kin...   
36009  [right, thank, recover, spiritual, well, fairn...   
9265   [support, happy, defeated, immense, like, rea